In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
ticker = "AAPL"
data = yf.download(ticker, start="2023-01-01", end="2025-01-01")

[*********************100%***********************]  1 of 1 completed


In [14]:
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

In [15]:
data.head()

Price,Close,High,Low,Open,Volume,MA,STD,Upper_Band,Lower_Band,Market_Return,Signal
Date,,,,,,,,,,,
2023-01-03,122.982719,128.715417,122.097738,128.105769,112117500,NaN,NaN,NaN,NaN,NaN,0
2023-01-04,124.251205,126.512824,122.992568,124.772359,89113600,NaN,NaN,NaN,NaN,0.010314,0
2023-01-05,122.933548,125.637653,122.677892,125.008335,80962700,NaN,NaN,NaN,NaN,-0.010605,0
2023-01-06,127.456787,128.115604,122.805730,123.907041,87754700,NaN,NaN,NaN,NaN,0.036794,0
2023-01-09,127.977913,131.183516,127.722257,128.292580,70790800,NaN,NaN,NaN,NaN,0.004089,0


In [16]:
window = 20
data['MA'] = data['Close'].rolling(window=window).mean()
data['STD'] = data['Close'].rolling(window=window).std()

Create Bollinger bands (2 SD).

In [17]:
data['Upper_Band'] = data['MA'] + (2 * data['STD'])
data['Lower_Band'] = data['MA'] - (2 * data['STD'])

Generate Trading Signals (Buy when below lower band, Sell when above upper band)

In [18]:
data['Signal'] = 0
# Buy Signal (1): Price is oversold
data.loc[data['Close'] < data['Lower_Band'], 'Signal'] = 1
# Sell Signal (-1): Price is overbought
data.loc[data['Close'] > data['Upper_Band'], 'Signal'] = -1

Calculate returns.

In [19]:
# Calculate daily percent change of the stock
data['Market_Return'] = data['Close'].pct_change()
# Shift signal by 1 day to avoid "look-ahead bias" (buying at today's close based on today's close)
data['Strategy_Return'] = data['Market_Return'] * data['Signal'].shift(1)

In [20]:
data['Cumulative_Market'] = (1 + data['Market_Return']).cumprod()
data['Cumulative_Strategy'] = (1 + data['Strategy_Return']).cumprod()

In [21]:
final_market = data['Cumulative_Market'].iloc[-1] - 1
final_strategy = data['Cumulative_Strategy'].iloc[-1] - 1
print(f"AAPL Buy & Hold Return: {final_market:.2%}")
print(f"Mean Reversion Strategy Return: {final_strategy:.2%}")

AAPL Buy & Hold Return: 102.33%
Mean Reversion Strategy Return: 2.97%


In [22]:
data.tail()

Price,Close,High,Low,Open,Volume,MA,STD,Upper_Band,Lower_Band,Market_Return,Signal,Strategy_Return,Cumulative_Market,Cumulative_Strategy
Date,,,,,,,,,,,,,,
2024-12-24,256.560822,256.570737,253.669277,253.868019,23234700,244.729901,6.506863,257.743627,231.716175,0.011478,0,0.0,2.086153,1.029664
2024-12-26,257.375549,258.448710,255.994390,256.550832,27237100,245.920293,6.535021,258.990335,232.850251,0.003176,0,0.0,2.092778,1.029664
2024-12-27,253.967361,257.057633,251.453425,256.193131,42355300,246.946735,6.066789,259.080312,234.813158,-0.013242,0,-0.0,2.065065,1.029664
2024-12-30,250.598877,251.890627,249.158085,250.628685,35557500,247.685513,5.515548,258.716609,236.654418,-0.013263,0,-0.0,2.037676,1.029664
2024-12-31,248.830200,251.672044,247.846480,250.837380,39480700,248.223576,5.031720,258.287016,238.160135,-0.007058,0,-0.0,2.023294,1.029664


Lesson: mean reversion underperforms in steady bull markets.